# Window=60 — Full Pipeline (Adaptive Threshold + Incremental Learning + Ablation)

**Purpose**: `window_size_ablation.ipynb` only tested the base VAE at
`WINDOW_SIZE=60` (F1 0.730 on `cc1_test`, but a weak 0.103 on `drift_cc2`).
This notebook checks whether the SAME two mechanisms that helped the deployed
(window=30) model recover drift performance — the blended adaptive threshold
and KS-test-triggered incremental learning — do the same for window=60, and
produces the same ablation table (`VAE alone` -> `+ Adaptive Threshold` ->
`Full Model`) as `final_comparison.ipynb`, so it's directly comparable.

Reuses the already-trained window=60 model and PCA
(`experiments/vae_cc1_window60.pt`, `experiments/cc1_pca_window60.pkl`) —
no retraining needed for this part.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle, joblib, os, copy
from collections import deque
from scipy import stats
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                              recall_score, f1_score, precision_recall_curve, confusion_matrix)

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
MODEL_DIR = os.path.join(BASE, 'models')
OUT_DIR   = os.path.join(BASE, 'experiments')
WINDOW_SIZE = 60
STRIDE = 1

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache',
]
SPLIT_FILES = {
    'cc1_train': 'cc1_train.csv', 'cc1_val': 'cc1_val.csv', 'cc1_test': 'cc1_test.csv',
    'drift_cc2': 'drift_complex_case2.csv',
}
ALL_SETS = ['cc1_test', 'drift_cc2']
DRIFT_SETS = ['drift_cc2']

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

w60_saved = pickle.load(open(os.path.join(OUT_DIR, 'window_size_60_results.pkl'), 'rb'))
meta60 = w60_saved['model_meta']
CLIP = meta60['clip']

base_model = VAE(meta60['input_dim'], meta60['hidden1'], meta60['hidden2'], meta60['latent_dim'])
base_model.load_state_dict(torch.load(os.path.join(OUT_DIR, 'vae_cc1_window60.pt'), map_location='cpu'))
base_model.eval()

pca_bundle = joblib.load(os.path.join(OUT_DIR, 'cc1_pca_window60.pkl'))
pca = pca_bundle['pca']

print(f'Loaded window=60 model: input_dim={meta60["input_dim"]}, latent_dim={meta60["latent_dim"]}, beta_max={meta60["beta_max"]}')
print(f'mu_train={meta60["mu_train"]:.5f}  sigma_train={meta60["sigma_train"]:.5f}  val_p99={meta60["val_p99"]:.5f}')

Loaded window=60 model: input_dim=48, latent_dim=32, beta_max=0.01
mu_train=0.28407  sigma_train=0.24760  val_p99=1.44733


## Step 1 — Rebuild windows at size 60 WITH per-window `cmdb_id`/timestamp tracked

(the original ablation only saved `X`/`y`/`ft`, not per-window container
identity — needed here for the per-container adaptive threshold and true
chronological streaming.)

In [2]:
def build_windows_full(df, feature_cols, window_size, stride):
    data_arr_all, cmdb_all, ts_all, y_all, ft_all = [], [], [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        data_arr = g[feature_cols].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ftypes = g['failure_type'].values.astype(object)
        ts = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i+window_size].any():
                continue
            data_arr_all.append(data_arr[i:i+window_size])
            cmdb_all.append(cmdb_id)
            ts_all.append(ts[i+window_size-1])
            y_all.append(int(labels[i:i+window_size].any()))
            w_types = sorted({t for t in ftypes[i:i+window_size] if isinstance(t, str)})
            ft_all.append(','.join(w_types) if w_types else None)
    X = np.stack(data_arr_all)
    return X, np.array(cmdb_all), np.array(ts_all), np.array(y_all, dtype=np.int64), np.array(ft_all, dtype=object)

raw_windows = {}
for name, fname in SPLIT_FILES.items():
    d = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    X, cmdb, ts, y, ft = build_windows_full(d, FEATURE_COLS, WINDOW_SIZE, STRIDE)
    X_flat = X.reshape(len(X), -1)
    X_pca = np.clip(pca.transform(X_flat), -CLIP, CLIP).astype(np.float32)
    raw_windows[name] = {'X': X_pca, 'cmdb': cmdb, 'ts': ts, 'y': y, 'ft': ft}
    print(f'  {name:10s}: {len(y):>7,} windows  ({int(y.sum()):,} anomalies)')

# Sanity check vs the earlier ablation's saved eval (same window/PCA config, should match exactly)
with torch.no_grad():
    check_mse = base_model.anomaly_score(torch.from_numpy(raw_windows['cc1_test']['X'])).numpy()
print(f'\nSanity check: recomputed mu on cc1_test should be close to earlier run - mean={check_mse.mean():.5f}')

  cc1_train : 152,456 windows  (0 anomalies)
  cc1_val   :  20,763 windows  (0 anomalies)
  cc1_test  :  43,375 windows  (256 anomalies)
  drift_cc2 :  76,167 windows  (1,080 anomalies)

Sanity check: recomputed mu on cc1_test should be close to earlier run - mean=0.32821


## Step 2 — Baseline VAE-alone evaluation (confirm consistency with `window_size_ablation.ipynb`)

In [3]:
VAL_P99 = meta60['val_p99']

def evaluate(scores, y_true, threshold):
    pred = (scores > threshold).astype(int)
    p, r, _ = precision_recall_curve(y_true, scores)
    f1s = 2 * p * r / (p + r + 1e-12)
    oracle_f1 = float(f1s[np.argmax(f1s)])
    return {
        'auc_roc': roc_auc_score(y_true, scores), 'auc_pr': average_precision_score(y_true, scores),
        'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0), 'oracle_f1': oracle_f1,
    }

vae_alone_results = {}
mse_cache = {}
for name in ALL_SETS:
    with torch.no_grad():
        mse = base_model.anomaly_score(torch.from_numpy(raw_windows[name]['X'])).numpy()
    mse_cache[name] = mse
    vae_alone_results[name] = evaluate(mse, raw_windows[name]['y'], VAL_P99)
    r = vae_alone_results[name]
    print(f'{name:12s} VAE alone   PR-AUC={r["auc_pr"]:.4f}  F1={r["f1"]:.3f}  Precision={r["precision"]:.3f}  Recall={r["recall"]:.3f}')

cc1_test     VAE alone   PR-AUC=0.9178  F1=0.912  Precision=0.986  Recall=0.848
drift_cc2    VAE alone   PR-AUC=0.3307  F1=0.155  Precision=0.086  Recall=0.800


## Step 3 — Adaptive blended threshold (calibrate `PRIOR_STRENGTH` fresh on `cc1_val` at window=60)

In [4]:
K_ADAPTIVE, BUFFER_SIZE = 3.0, 500
GLOBAL_MEAN, GLOBAL_STD = meta60['mu_train'], meta60['sigma_train']

def run_blended(mse_arr, cmdb_arr, prior_strength, buffer_size=BUFFER_SIZE, k=K_ADAPTIVE,
                global_mean=GLOBAL_MEAN, global_std=GLOBAL_STD):
    n = len(mse_arr)
    preds = np.zeros(n, dtype=np.int64)
    buffers = {}
    for i in range(n):
        cid = cmdb_arr[i]
        buf = buffers.setdefault(cid, deque(maxlen=buffer_size))
        n_local = len(buf)
        w = n_local / (n_local + prior_strength)
        if n_local == 0:
            lm, ls = global_mean, global_std
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        t = (w * lm + (1 - w) * global_mean) + k * (w * ls + (1 - w) * global_std)
        is_anom = mse_arr[i] > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse_arr[i])
    return preds

with torch.no_grad():
    mse_val = base_model.anomaly_score(torch.from_numpy(raw_windows['cc1_val']['X'])).numpy()

PRIOR_CANDIDATES = [100, 500, 2000, 10000, 50000]
print(f'{"prior_strength":>14s} {"FPR on cc1_val":>16s}')
prior_fpr = {}
for ps in PRIOR_CANDIDATES:
    preds = run_blended(mse_val, raw_windows['cc1_val']['cmdb'], prior_strength=ps)
    fpr = preds.mean()
    prior_fpr[ps] = fpr
    print(f'{ps:14d} {fpr*100:15.2f}%')

BEST_PRIOR = min(PRIOR_CANDIDATES, key=lambda ps: abs(prior_fpr[ps] - 0.01))
print(f'\nChosen PRIOR_STRENGTH = {BEST_PRIOR}')

prior_strength   FPR on cc1_val
           100            2.28%
           500            2.03%
          2000            1.92%
         10000            1.90%
         50000            1.91%

Chosen PRIOR_STRENGTH = 10000


In [5]:
adaptive_results = {}
for name in ALL_SETS:
    preds = run_blended(mse_cache[name], raw_windows[name]['cmdb'], prior_strength=BEST_PRIOR)
    y = raw_windows[name]['y']
    p = precision_score(y, preds, zero_division=0)
    r = recall_score(y, preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    adaptive_results[name] = {'precision': p, 'recall': r, 'f1': f1}
    print(f'{name:12s} + Adaptive   Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}   '
          f'(vs VAE alone F1={vae_alone_results[name]["f1"]:.3f})')

cc1_test     + Adaptive   Precision=0.665  Recall=0.906  F1=0.767   (vs VAE alone F1=0.912)
drift_cc2    + Adaptive   Precision=0.059  Recall=0.893  F1=0.110   (vs VAE alone F1=0.155)


## Step 4 — Incremental learning (KS-test-triggered fine-tuning), true chronological order

In [6]:
REFIT_INTERVAL, FT_BUFFER_SIZE, FT_LR, FT_EPOCHS, KS_ALPHA = 5000, 2000, 1e-4, 5, 0.001

rng = np.random.default_rng(42)
ref_idx = rng.choice(len(raw_windows['cc1_train']['X']), size=5000, replace=False)
REF_SAMPLE = raw_windows['cc1_train']['X'][ref_idx]
with torch.no_grad():
    REFERENCE_MSE = base_model.anomaly_score(torch.from_numpy(REF_SAMPLE)).numpy()
print(f'Frozen KS-test reference: {len(REFERENCE_MSE):,} cc1_train windows (window=60, scored by the original model).')

def run_stream(model, X_stream, cmdb_stream, incremental_learning, reference_mse):
    model = copy.deepcopy(model)
    opt = torch.optim.Adam(model.parameters(), lr=FT_LR)
    preds = np.zeros(len(X_stream), dtype=np.int64)
    threshold_buffers = {}
    ft_pool = deque(maxlen=FT_BUFFER_SIZE)
    n_finetunes, finetune_events = 0, []

    for i in range(len(X_stream)):
        x_i = X_stream[i]
        cid = cmdb_stream[i]
        x_t = torch.from_numpy(x_i).unsqueeze(0)
        with torch.no_grad():
            mse = model.anomaly_score(x_t).item()

        buf = threshold_buffers.setdefault(cid, deque(maxlen=BUFFER_SIZE))
        n_local = len(buf)
        w = n_local / (n_local + BEST_PRIOR)
        if n_local == 0:
            lm, ls = GLOBAL_MEAN, GLOBAL_STD
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        t = (w * lm + (1 - w) * GLOBAL_MEAN) + K_ADAPTIVE * (w * ls + (1 - w) * GLOBAL_STD)

        is_anom = mse > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse)
            ft_pool.append(x_i)

        if incremental_learning and (i + 1) % REFIT_INTERVAL == 0 and len(ft_pool) >= FT_BUFFER_SIZE // 2:
            with torch.no_grad():
                recent_mse = model.anomaly_score(torch.from_numpy(np.stack(list(ft_pool)))).numpy()
            _, p_value = stats.ks_2samp(reference_mse, recent_mse)
            if p_value < KS_ALPHA:
                Xb = torch.from_numpy(np.stack(list(ft_pool)))
                model.train()
                for _ in range(FT_EPOCHS):
                    opt.zero_grad()
                    recon, mu, logvar = model(Xb)
                    recon_loss = nn.functional.mse_loss(recon, Xb, reduction='mean')
                    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                    (recon_loss + meta60['beta_max'] * kl).backward()
                    opt.step()
                model.eval()
                n_finetunes += 1
                finetune_events.append(i + 1)
    return preds, n_finetunes, finetune_events

full_model_results = {}
for name in ALL_SETS:
    order = np.argsort(raw_windows[name]['ts'], kind='stable')
    X_stream = raw_windows[name]['X'][order]
    cmdb_stream = raw_windows[name]['cmdb'][order]
    y_stream = raw_windows[name]['y'][order]

    preds_full, n_ft, ft_events = run_stream(base_model, X_stream, cmdb_stream, incremental_learning=True, reference_mse=REFERENCE_MSE)
    p = precision_score(y_stream, preds_full, zero_division=0)
    r = recall_score(y_stream, preds_full, zero_division=0)
    f1 = f1_score(y_stream, preds_full, zero_division=0)
    full_model_results[name] = {'precision': p, 'recall': r, 'f1': f1, 'n_finetunes': n_ft, 'finetune_events': ft_events}
    print(f'{name:12s} Full Model  Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}  (fine-tunes: {n_ft} at {ft_events})')

Frozen KS-test reference: 5,000 cc1_train windows (window=60, scored by the original model).
cc1_test     Full Model  Precision=0.667  Recall=0.906  F1=0.768  (fine-tunes: 6 at [5000, 15000, 20000, 25000, 30000, 35000])
drift_cc2    Full Model  Precision=0.061  Recall=0.894  F1=0.115  (fine-tunes: 15 at [5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000, 45000, 50000, 55000, 60000, 65000, 70000, 75000])


## Step 5 — Ablation table: window=60 vs. deployed window=30 (`final_comparison.ipynb`'s numbers), same structure

In [7]:
deployed_ablation = {
    'cc1_test':  {'vae_alone': 0.618, 'adaptive': 0.623, 'full_model': 0.634},
    'drift_cc2': {'vae_alone': 0.276, 'adaptive': 0.347, 'full_model': 0.391},
}

print(f'{"set":12s} {"config":14s} {"window=30 (deployed)":>22s} {"window=60":>11s}')
for name in ALL_SETS:
    print(f'{name:12s} {"VAE alone":14s} {deployed_ablation[name]["vae_alone"]:22.3f} {vae_alone_results[name]["f1"]:11.3f}')
    print(f'{name:12s} {"+ Adaptive":14s} {deployed_ablation[name]["adaptive"]:22.3f} {adaptive_results[name]["f1"]:11.3f}')
    print(f'{name:12s} {"Full Model":14s} {deployed_ablation[name]["full_model"]:22.3f} {full_model_results[name]["f1"]:11.3f}')
    print()

set          config           window=30 (deployed)   window=60
cc1_test     VAE alone                       0.618       0.912
cc1_test     + Adaptive                      0.623       0.767
cc1_test     Full Model                      0.634       0.768

drift_cc2    VAE alone                       0.276       0.155
drift_cc2    + Adaptive                      0.347       0.110
drift_cc2    Full Model                      0.391       0.115



## Step 6 — Save

In [8]:
save_results = {
    'window_size': WINDOW_SIZE,
    'vae_alone': vae_alone_results,
    'adaptive_blended': {'prior_strength': BEST_PRIOR, 'results': adaptive_results},
    'full_model': full_model_results,
    'deployed_comparison': deployed_ablation,
}
out_path = os.path.join(OUT_DIR, 'window_size_60_full_pipeline_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\window_size_60_full_pipeline_results.pkl


## How to read this

This mirrors `final_comparison.ipynb`'s ablation structure exactly, just at
`WINDOW_SIZE=60` instead of 30. The honest verdict is Step 5's table: does
adaptive thresholding + incremental learning close enough of window=60's
`drift_cc2` gap to make it competitive with (or better than) the deployed
window=30 pipeline on BOTH sets simultaneously — not just `cc1_test` alone?